## Pré-processamento inicial de um arquivo EEG

In [ ]:
# Instalação e Importação do MNE
%pip install mne
import mne
import os
%pip install mne-qt-browser PyQt5
mne.viz.set_browser_backend('qt')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Leitura de um arquivo .gdf do dataset
ficheiro = 'data/ID1.gdf' # Altere aqui para rodar outro arquivo
raw = mne.io.read_raw_gdf(ficheiro, preload=True)

Extracting GDF parameters from data/ID1.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, P8, Fz, Cz, Pz, M1, M2, AFz, CPz, POz
Creating raw.info structure...
Reading 0 ... 151455  =      0.000 ...   605.820 secs...


In [11]:
# Dados do raw data
print(raw.info)

raw.plot(title='Dados Bruto')

<Info | 8 non-empty values
 bads: []
 ch_names: FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, ...
 chs: 24 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 24
 projs: []
 sfreq: 250.0 Hz
 subject_info: <subject_info | his_id: X, last_name: >
>


In [8]:
import glob

# Definir as pastas de entrada e saída
pasta_entrada = 'data'
pasta_saida = 'cleaned'

# Criar a pasta 'cleaned' caso ela ainda não exista
os.makedirs(pasta_saida, exist_ok=True)

# Encontrar todos os arquivos .gdf na pasta de entrada
# O glob vai retornar uma lista com os caminhos, ex: ['data/ID1.gdf', 'data/ID2.gdf', ...]
arquivos_gdf = glob.glob(os.path.join(pasta_entrada, '*.gdf'))

# Verificar se encontrou arquivos
if not arquivos_gdf:
    print(f"Nenhum arquivo .gdf encontrado na pasta '{pasta_entrada}'.")
else:
    print(f"Foram encontrados {len(arquivos_gdf)} arquivos para processar.\n")

# Laço de repetição para processar cada arquivo
for ficheiro in arquivos_gdf:
    # Extrair apenas o nome do arquivo sem a extensão e sem a pasta (ex: 'ID1')
    nome_base = os.path.splitext(os.path.basename(ficheiro))[0]
    
    # Leitura do arquivo .gdf
    raw = mne.io.read_raw_gdf(ficheiro, preload=True)
    raw_proc = raw.copy()
    
    # Definir a montagem dos elétrodos
    montagem = mne.channels.make_standard_montage('standard_1020')
    raw_proc.set_montage(montagem, match_case=False, on_missing='warn')
    
    # Filtragem de Frequências (Band-pass filter de 1.0 a 40.0 Hz)
    raw_proc.filter(l_freq=1.0, h_freq=40.0)
    
    # Filtro Notch (remover o ruído da rede elétrica - 60 Hz)
    frequencia_rede = 60.0 
    raw_proc.notch_filter(freqs=frequencia_rede)
    
    # Re-referenciação
    raw_proc.set_eeg_reference('average', projection=True)
    raw_proc.apply_proj()
    
    # Configurar o ICA (usando 15 componentes, já que temos 24 canais)
    ica = mne.preprocessing.ICA(n_components=15, random_state=97, max_iter='auto')
    
    # Ajustar o ICA nos dados já filtrados
    ica.fit(raw_proc)
    
    # Encontrar automaticamente os componentes relacionados ao piscar de olhos
    # Usamos o canal 'FP1' como referência para captar o movimento ocular
    bads_eog, scores_eog = ica.find_bads_eog(raw_proc, ch_name='FP1')
    
    # Marcar esses componentes como "ruins" para serem removidos
    ica.exclude = bads_eog
    print(f"Componentes removidos automaticamente pelo ICA: {bads_eog}")
    
    # Aplicar o ICA para limpar o sinal (isso altera o raw_proc de forma definitiva)
    ica.apply(raw_proc)

    # Definir o caminho onde o arquivo será salvo dinamicamente
    nome_arquivo_salvar = f"{nome_base}_preproc_eeg.fif"
    caminho_salvar = os.path.join(pasta_saida, nome_arquivo_salvar)
    
    # Guardar os dados processados
    raw_proc.save(caminho_salvar, overwrite=True)

print("Processamento de todos os arquivos concluído!")

Foram encontrados 37 arquivos para processar.

Extracting GDF parameters from data\ID0.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, P8, Fz, Cz, Pz, M1, M2, AFz, CPz, POz
Creating raw.info structure...
Reading 0 ... 149951  =      0.000 ...   599.804 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 825 samples (3.300 s)

Filtering raw data in 1 contiguous seg

In [9]:
# Leitura de um arquivo .fif do cleaned
ficheiro_preproc = 'cleaned/ID1_preproc_eeg.fif' # Altere aqui para rodar outro arquivo
raw_preproc = mne.io.read_raw_fif(ficheiro_preproc, preload=True)

Opening raw data file cleaned/ID1_preproc_eeg.fif...
    Read a total of 1 projection items:
        Average EEG reference (1 x 24) active
    Range : 0 ... 151455 =      0.000 ...   605.820 secs
Ready.
Reading 0 ... 151455  =      0.000 ...   605.820 secs...


In [10]:
# Dados do raw_preproc data
print(raw_preproc.info)

raw_preproc.plot(title='Dados Pré-processados')

<Info | 12 non-empty values
 bads: []
 ch_names: FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, ...
 chs: 24 EEG
 custom_ref_applied: False
 dig: 27 items (3 Cardinal, 24 EEG)
 file_id: 4 items (dict)
 highpass: 1.0 Hz
 lowpass: 40.0 Hz
 meas_date: unspecified
 meas_id: 4 items (dict)
 nchan: 24
 projs: Average EEG reference: on
 sfreq: 250.0 Hz
 subject_info: <subject_info | his_id: X>
>
